In [1]:
from datetime import datetime
import pandas as pd
import os 
from IPython.display import display


## import helper

In [2]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules
from config import gam_info

from functions import execute_sql_query
import test_functions

In [3]:
# country
country_codes = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='CountryID', 
                              keep_default_na=False)

# week 
week_tester = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='GAM Period')
week_tester['w/c'] = pd.to_datetime(week_tester['w/c'])

# podcast details
podcast_details = pd.read_excel(f"../../{gam_info['lookup_file']}", 
                                     sheet_name='Podcast').dropna(how='all')

### RUN TESTS
test_functions.test_lookup_files(country_codes, ['PlaceID'], [0,1,2],
                                 test_step = "lookup files - ensuring country codes are correct")

test_functions.test_lookup_files(week_tester, ['w/c'], [3,4,5], 
                                 test_step = "lookup files - ensuring week tester is correct")


✅ Test 0 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test 1 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test 2 passed: No missing values in lookup.
...updating logbook...

✅ Test 3 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test 4 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test 5 passed: No missing values in lookup.
...updating logbook...



## helper functions

In [4]:
def get_formatted_values(df, query_type):
    values = df[df['Query Type'] == query_type]['Value'].tolist()
    return ', '.join(f"'{value}'" for value in values)

def generate_when_clauses(df):
    when_clause_brands = ''
    when_clause_programmes = ''
    for index, row in df.iterrows():
        if row['Query Type'] == 'brand_id':
            when_clause_brands += f"           WHEN vmb.master_brand_id = '{row['Value']}' THEN '{row['Service']}'\n"
        elif row['Query Type'] == 'program_title':
            when_clause_programmes += f"           WHEN vmb.programme_title = '{row['Value']}' THEN '{row['Service']}'\n"
    return when_clause_brands, when_clause_programmes

In [5]:
def execute_sql_query_with_output(sql_query, file_timeinfo, output_filename):
    df = execute_sql_query(sql_query)
    if df is not None:
        print(df.head())

    
    file_path = f"../data/raw/podcast"
    os.makedirs(file_path, exist_ok=True)
    df.to_csv(f"{file_path}/{file_timeinfo}_{output_filename}.csv", index=False)
    
    return df

# ingestion

## test I find all the programes

In [6]:
# Cell 1: Construct and execute the first query
formatted_brand_ids = get_formatted_values(podcast_details, 'brand_id')
formatted_program_titles = get_formatted_values(podcast_details, 'program_title')

sql_query_1 = f"""SELECT 
                    vmb.master_brand_id,
                    vmb.programme_title,
                    COUNT(prd.version_id) AS entry_count
                FROM 
                    {gam_info['podcast_sql_table']} prd
                INNER JOIN 
                    redshiftdb.prez.scv_vmb vmb 
                    ON prd.version_id = vmb.version_id
                WHERE 
                    prd.date_utc BETWEEN '{gam_info['w/c_start']}' AND '{gam_info['weekEnding_end']}'
                    AND (
                        vmb.master_brand_id IN ({formatted_brand_ids})
                    OR 
                        vmb.programme_title IN ({formatted_program_titles})
                    )
                GROUP BY 
                    vmb.master_brand_id, vmb.programme_title
                HAVING 
                    COUNT(prd.version_id) > 0
                    ;"""

test_df = execute_sql_query_with_output(sql_query_1, 
                                        gam_info['file_timeinfo'], 
                                        'podcast_test_finding_all_brandAndProgrammes')
################################### Testing ################################### 
test_step = 'sql returns for all programmes'
column_name = 'master_brand_id'
brand_ids = [title.strip("'") for title in formatted_brand_ids.split(", ")]
test_functions.test_filter_elements_returned(test_df, brand_ids, column_name, "1_POD_1", test_step)
column_name = 'programme_title'
program_titles = [title.strip("'") for title in formatted_program_titles.split(", ")]
test_functions.test_filter_elements_returned(test_df, program_titles, column_name, '1_POD_2', test_step)

################################### Testing ################################### 


     master_brand_id                        programme_title  entry_count
0  bbc_world_service          The Compass disUnited Kingdom          113
1  bbc_world_service                        Witness History      4409598
2  bbc_russian_radio     Радиоархив Русской службы Би-би-си         2807
3  bbc_world_service  Discovery The Truth About Parkinson's         1812
4  bbc_world_service       Discovery A Tale of Two Axolotls          706
...testing master_brand_id...
✅ Test 1_POD_1 passed: everything found!
...updating logbook...

...testing programme_title...
❌ Test 1_POD_2 failed: not all elements from lookup were retrieved in dataset - decide if they are really missing or if these accounts are inactive 
...updating logbook...



## Individual language services 

In [7]:
#Construct and execute the second query
when_clause_brands, when_clause_programmes = generate_when_clauses(podcast_details)

sql_query_2 = f"""
    SELECT DATE_TRUNC('week', prd.date_utc) AS week,
           CASE
               {when_clause_programmes}
               ELSE CASE
                   {when_clause_brands}
                   ELSE 'Unknown'
               END
           END AS service,
           country,
               COUNT(DISTINCT CONCAT(prd.ip, prd.useragent)) AS uniques,
           COUNT(DISTINCT prd.ip) AS old_uniques,
           COUNT(*) AS downloads
    FROM {gam_info['podcast_sql_table']} prd
    INNER JOIN redshiftdb.prez.scv_vmb vmb 
        ON prd.version_id = vmb.version_id
    WHERE 
        prd.date_utc BETWEEN '{gam_info['w/c_start']}' AND '{gam_info['weekEnding_end']}'
        AND (
        vmb.master_brand_id IN ({formatted_brand_ids})
        OR 
        vmb.programme_title IN ({formatted_program_titles})
        )
    GROUP BY week, service, country
;
"""

podcast_raw = execute_sql_query_with_output(sql_query_2, gam_info['file_timeinfo'], 'podcast_individualLanguages_redshift_extract')
podcast_raw.rename(columns={'week': 'w/c'}, inplace=True)
#display(podcast_raw.sample())

################################### Testing ################################### 
test_step = 'sql returns for individual language services'
test_functions.podcast_test_services_in_results(podcast_raw, podcast_details, '1_POD_3', test_step)
test_functions.podcast_check_unknown_services(podcast_raw,'1_POD_4', test_step)

# weeks there? 
test_functions.test_weeks_presence_per_account('w/c', 'service', podcast_raw, week_tester,
                                                '1_POD_5', test_step)
################################### Testing ################################### 

        week                      service country  uniques  old_uniques  \
0 2025-06-16  * BBC World Service English      tw    43950        39867   
1 2025-05-12             Learning English      gb    21160        17088   
2 2025-05-12  * BBC World Service English      es    40696        33794   
3 2025-12-01             Learning English      tw    37561        33860   
4 2025-12-01                      Persian      it      158          127   

   downloads  
0     149998  
1      52428  
2     111615  
3      79624  
4        268  
✅ Pass - All services are listed in the SQL results.
...updating logbook...

✅ Pass - No 'Unknown' services found in the SQL results.
...updating logbook...

❌ Test 1_POD_5 failed: Missing completed weeks detected.
...updating logbook...



## BBC World Service Languages


In [8]:
# Cell 3: Construct and execute the third query with WSL filter
total_wsl = podcast_details[podcast_details['* BBC World Service Languages'] == True]
formatted_brand_ids = get_formatted_values(total_wsl, 'brand_id')
formatted_program_titles = get_formatted_values(total_wsl, 'program_title')

sql_query_3 = f"""
    SELECT DATE_TRUNC('week', prd.date_utc) AS week,
           CASE
               WHEN 
                    vmb.master_brand_id IN ({formatted_brand_ids})
                    OR 
                    vmb.programme_title IN ({formatted_program_titles})
                    THEN '* BBC World Service Languages'
           END AS service,
           country,
           COUNT(DISTINCT CONCAT(prd.ip, prd.useragent)) AS uniques,
           COUNT(DISTINCT prd.ip) AS old_uniques,
           COUNT(*) AS downloads
    FROM {gam_info['podcast_sql_table']} prd
    INNER JOIN redshiftdb.prez.scv_vmb vmb 
        ON prd.version_id = vmb.version_id
    WHERE 
        prd.date_utc BETWEEN '{gam_info['w/c_start']}' AND '{gam_info['weekEnding_end']}'
        AND (
        vmb.master_brand_id IN ({formatted_brand_ids})
        OR 
        vmb.programme_title IN ({formatted_program_titles})
        )
    GROUP BY week, service, country
;
"""

podcast_total_wsl = execute_sql_query_with_output(sql_query_3, gam_info['file_timeinfo'], 'podcast_totalWSL_redshift_extract')
podcast_total_wsl.rename(columns={'week': 'w/c'}, inplace=True)
#display(podcast_total_wsl.sample())
#wpodcast_test_services_in_results(podcast_total_wsl, total_wsl)
################################### Testing ################################### 
test_step = 'sql returns for WSL'
test_functions.podcast_check_unknown_services(podcast_total_wsl,'1_POD_6', test_step)

test_functions.test_weeks_presence_per_account('w/c', 'service', podcast_total_wsl, week_tester,
                                                '1_POD_7', test_step)
################################### Testing ################################### 

        week                        service country  uniques  old_uniques  \
0 2025-03-31  * BBC World Service Languages      sk     2676         2256   
1 2025-11-10  * BBC World Service Languages      de    54700        41343   
2 2025-11-03  * BBC World Service Languages      hr      269          240   
3 2025-07-21  * BBC World Service Languages      gw       11           11   
4 2025-07-21  * BBC World Service Languages      az     6215         5151   

   downloads  
0       6048  
1     117465  
2        614  
3         82  
4       9317  
✅ Pass - No 'Unknown' services found in the SQL results.
...updating logbook...

✅ Test 1_POD_7 passed: All completed weeks present for each account.
...updating logbook...



## BBC World Service 

In [9]:
# Filter the DataFrame based on '* BBC World Service'
total_ws = podcast_details[podcast_details['* BBC World Service'] == True]

# Generate the formatted brand IDs and program titles
formatted_brand_ids = get_formatted_values(total_ws, 'brand_id')
formatted_program_titles = get_formatted_values(total_ws, 'program_title')

# Construct the SQL query
sql_query = f"""
    SELECT DATE_TRUNC('week', prd.date_utc) AS week,
           CASE
               WHEN 
                    vmb.master_brand_id IN ({formatted_brand_ids})
                    OR 
                    vmb.programme_title IN ({formatted_program_titles})
                    THEN '* BBC World Service'
                WHEN country!='gb' 
                    THEN 'UKPS'
           ELSE 'UKPS GB'      
           END AS service,
           country,
           COUNT(DISTINCT CONCAT(prd.ip, prd.useragent)) AS uniques,
           COUNT(DISTINCT prd.ip) AS old_uniques,
           COUNT(*) AS downloads
    FROM {gam_info['podcast_sql_table']} prd
    INNER JOIN redshiftdb.prez.scv_vmb vmb 
        ON prd.version_id = vmb.version_id
    WHERE 
        prd.date_utc BETWEEN '{gam_info['w/c_start']}' AND '{gam_info['weekEnding_end']}'
    GROUP BY week, service, country
    HAVING service != 'UKPS GB'
;
"""

podcast_total_ws = execute_sql_query_with_output(sql_query, gam_info['file_timeinfo'], 'podcast_totalWS_redshift_extract')
podcast_total_ws.rename(columns={'week': 'w/c'}, inplace=True)
#display(podcast_total_wsl.sample())

################################### Testing ################################### 
test_step = 'sql returns for WS'
#podcast_test_services_in_results(podcast_total_ws, total_ws)
test_functions.podcast_check_unknown_services(podcast_total_ws, '1_POD_8', test_step)

test_functions.test_weeks_presence_per_account('w/c', 'service', podcast_total_wsl, week_tester,
                                                '1_POD_9', test_step)
################################### Testing ################################### 

        week              service country  uniques  old_uniques  downloads
0 2025-05-12  * BBC World Service      us   582819       431589    1912925
1 2025-10-06                 UKPS      kr     6205         5363      18067
2 2025-12-01  * BBC World Service      hk    52246        35369     150585
3 2025-09-08  * BBC World Service      nz    36395        28407     132733
4 2025-10-13  * BBC World Service      tr    53306        48671     134122
✅ Pass - No 'Unknown' services found in the SQL results.
...updating logbook...

✅ Test 1_POD_9 passed: All completed weeks present for each account.
...updating logbook...



## all BBC 

In [10]:
# Filter the DataFrame based on '* BBC World Service'
all_bbc = podcast_details[podcast_details['* All BBC'] == True]

# Generate the formatted brand IDs and program titles
formatted_brand_ids = get_formatted_values(all_bbc, 'brand_id')
formatted_program_titles = get_formatted_values(all_bbc, 'program_title')

# Construct the SQL query
sql_query = f"""
    SELECT DATE_TRUNC('week', prd.date_utc) AS week,
           CASE
               WHEN 
                    vmb.master_brand_id IN ({formatted_brand_ids})
                    OR 
                    vmb.programme_title IN ({formatted_program_titles})
                    THEN '* All BBC'
               WHEN country!='gb' THEN '* All BBC'  
           END AS service,
           country,
           COUNT(DISTINCT CONCAT(prd.ip, prd.useragent)) AS uniques,
           COUNT(DISTINCT prd.ip) AS old_uniques,
           COUNT(*) AS downloads
    FROM {gam_info['podcast_sql_table']} prd
    INNER JOIN redshiftdb.prez.scv_vmb vmb 
        ON prd.version_id = vmb.version_id
    WHERE 
        prd.date_utc BETWEEN '{gam_info['w/c_start']}' AND '{gam_info['weekEnding_end']}'
    GROUP BY week, service, country
;
"""

podcast_total_allBBC = execute_sql_query_with_output(sql_query, gam_info['file_timeinfo'], 'podcast_allBBC_redshift_extract')
podcast_total_allBBC.rename(columns={'week': 'w/c'}, inplace=True)
#display(podcast_total_wsl.sample())

################################### Testing ################################### 
test_step = 'sql returns for all BBC'

#podcast_test_services_in_results(podcast_total_allBBC, all_bbc)
test_functions.podcast_check_unknown_services(podcast_total_allBBC, '1_POD_10', test_step)

test_functions.test_weeks_presence_per_account('w/c', 'service', podcast_total_allBBC, week_tester,
                                                '1_POD_11', test_step)

################################### Testing ################################### 

        week    service country  uniques  old_uniques  downloads
0 2025-05-05  * All BBC      sa    54184        48271      90976
1 2025-05-05  * All BBC      ua    19928        18132      40861
2 2025-11-03  * All BBC      kr    79533        59457     221118
3 2025-05-05  * All BBC      by     2526         2330       5974
4 2025-10-27  * All BBC      ng     9689         7344      22950
✅ Pass - No 'Unknown' services found in the SQL results.
...updating logbook...

✅ Test 1_POD_11 passed: All completed weeks present for each account.
...updating logbook...

